# 07 - Visualization Workbench
Scaffold embedded data science outputs inside the Banana notebook runner.

This notebook emits special output markers consumed by the in-page renderer:
- `BANANA_PLOTLY::` for full Plotly figure JSON (preferred)
- `BANANA_TABLE::` for tabular data
- `BANANA_VEGA::`, `BANANA_BOKEH::`, and `BANANA_WIDGET::` for runtime-option families that are surfaced but still gated
- `BANANA_GEO::`, `BANANA_NETWORK::`, and `BANANA_3D::` for specialized surfaces that are explicitly deferred

In [ ]:
import json
from js import XMLHttpRequest

API_BASE = 'https://api.banana.engineer'

def pp(obj):
    if isinstance(obj, str):
        try:
            obj = json.loads(obj)
        except Exception:
            print(obj)
            return
    print(json.dumps(obj, indent=2))

def call_endpoint(method, path, payload=None, headers=None):
    url = path if str(path).startswith('http') else f"{API_BASE}{path}"
    try:
        xhr = XMLHttpRequest.new()
        xhr.open(method.upper(), url, False)
        xhr.setRequestHeader('Accept', 'application/json')
        if headers:
            for k, v in headers.items():
                xhr.setRequestHeader(str(k), str(v))
        body = None
        if payload is not None:
            xhr.setRequestHeader('Content-Type', 'application/json')
            body = json.dumps(payload)
        xhr.send(body)
        return {'ok': 200 <= int(xhr.status) < 300, 'status': int(xhr.status), 'body': str(xhr.responseText)}
    except Exception as ex:
        return {'ok': False, 'status': None, 'body': str(ex)}

def get(path, **kw):
    return call_endpoint('GET', path, **kw)

print('viz_workbench_ready')

In [ ]:
# Bar chart scaffold (Plotly)
figure = {
    'data': [{'type': 'bar', 'x': ['banana', 'ripeness', 'chat', 'trucks'], 'y': [42, 31, 18, 25], 'marker': {'color': ['#3b82f6','#10b981','#f59e0b','#ef4444']}}],
    'layout': {'title': {'text': 'Requests by Endpoint Group'}, 'xaxis': {'title': 'group'}, 'yaxis': {'title': 'count'}, 'paper_bgcolor': 'rgba(0,0,0,0)', 'plot_bgcolor': 'rgba(0,0,0,0)'}
}
print('BANANA_PLOTLY::' + json.dumps(figure))

In [ ]:
# Line chart scaffold (Plotly scatter with lines)
figure = {
    'data': [{'type': 'scatter', 'mode': 'lines+markers', 'x': ['run-1','run-2','run-3','run-4','run-5'], 'y': [0.78, 0.82, 0.85, 0.87, 0.90], 'name': 'accuracy', 'line': {'color': '#3b82f6'}}],
    'layout': {'title': {'text': 'Model Accuracy Trend'}, 'xaxis': {'title': 'run'}, 'yaxis': {'title': 'accuracy', 'range': [0, 1]}, 'paper_bgcolor': 'rgba(0,0,0,0)', 'plot_bgcolor': 'rgba(0,0,0,0)'}
}
print('BANANA_PLOTLY::' + json.dumps(figure))

In [ ]:
# Table scaffold
table = {
    'columns': ['lane', 'samples', 'accuracy'],
    'rows': [
        ['banana', 512, 0.94],
        ['not-banana', 498, 0.91],
        ['ripeness', 321, 0.88]
    ]
}
print('BANANA_TABLE::' + json.dumps(table))

In [ ]:
# Live API -> table + bar chart (Plotly)
r = get('/swagger/v1/swagger.json')
if not r['ok']:
    print('status:', r['status'])
    print(r['body'])
else:
    spec_doc = json.loads(r['body'])
    counts = {}
    for path, methods in (spec_doc.get('paths') or {}).items():
        for method, details in methods.items():
            if method.lower() not in ('get','post','put','patch','delete','head','options'):
                continue
            group = (details.get('tags') or ['Ungrouped'])[0]
            counts[group] = counts.get(group, 0) + 1

    labels = sorted(counts.keys())
    values = [counts[g] for g in labels]

    table = {
        'columns': ['group', 'operation_count'],
        'rows': [[g, counts[g]] for g in labels]
    }
    figure = {
        'data': [{'type': 'bar', 'x': labels, 'y': values, 'marker': {'color': '#3b82f6'}}],
        'layout': {'title': {'text': 'OpenAPI Operations by Group'}, 'xaxis': {'title': 'group'}, 'yaxis': {'title': 'operations'}, 'paper_bgcolor': 'rgba(0,0,0,0)', 'plot_bgcolor': 'rgba(0,0,0,0)'}
    }

    print('BANANA_TABLE::' + json.dumps(table))
    print('BANANA_PLOTLY::' + json.dumps(figure))

In [ ]:
# Runtime-option families captured as explicit gates
runtime_gate = {
    'title': 'Vega-Lite contract sample',
    'summary': 'Declarative analytics specs are preserved and surfaced, but not rendered in the active wave.',
    'detail': 'The replacement DS page keeps these notebook contracts visible without widening the browser runtime prematurely.'
}
print('BANANA_VEGA::' + json.dumps(runtime_gate))
print('BANANA_BOKEH::' + json.dumps({
    'title': 'Bokeh runtime gate',
    'summary': 'Streaming and Bokeh bundle payloads remain gated runtime options.'
}))
print('BANANA_WIDGET::' + json.dumps({
    'title': 'Widget interop gate',
    'summary': 'ipywidgets-style controls remain gated until state replay and security boundaries are implemented.'
}))

In [ ]:
# Specialized notebook surfaces remain explicitly deferred
print('BANANA_GEO::' + json.dumps({
    'title': 'Geospatial surface deferred',
    'summary': 'Map and tile flows are explicitly deferred beyond the current release boundary.'
}))
print('BANANA_NETWORK::' + json.dumps({
    'title': 'Network graph surface deferred',
    'summary': 'Relationship views need a dedicated host and are not bundled into the default notebook runner.'
}))
print('BANANA_3D::' + json.dumps({
    'title': '3D surface deferred',
    'summary': 'WebGL-heavy notebook views remain outside the current DS release boundary.'
}))